# E1: streaming chat from a loopback server

D5 accepts token IDs. A chat client sends JSON messages and expects text. E1 connects those interfaces with a small OpenAI-compatible **Chat Completions subset**: JSON responses or server-sent events (SSE).

**Scope:** only `127.0.0.1` inside this Colab runtime. There is no public tunnel, authentication or deployment. Do not expose this teaching app publicly.

**Validation status:** E1's separate T4 run at source `cee6f49` passed all 114 D5/E1 tests on 2026-11-11; complete evidence is archived in `results/E1_streaming_server.json`. D5's complete `d013743` run is validated under its paging/greedy-parity precision contract; cross-precision scores are not claimed equivalent. This notebook deliberately uses **FP32**, one serving model and a small explicit KV pool.

Use a **fresh Colab T4**. Do not run this alongside the D5 models. Local validation is static only. The server runs only inside an async context and is stopped before results are saved.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd /content/tinyserve
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--index-url",
                "https://pypi.org/simple", "-e", ".[dev,server]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)

## 1. Tests before a real server

Run the D5 engine and E1 protocol/lifecycle tests first. The E1 tests check exact JSON/SSE shapes, Unicode reconstruction, EOS, invalid options, a 64 KiB request-body limit, bounded output queues, concurrent ownership, cancellation and explicit engine failure.

Most HTTP tests use an in-process ASGI transport. **ASGI** is the interface between our Python app and the HTTP server. That transport can buffer response data, so it cannot prove real incremental delivery. A separate test starts a real loopback socket, reads the first event, closes the client and verifies block release.

The D5 tests use small reference models; their result does not replace full-TinyLlama notebook evidence. The separately archived D5 run provides that evidence for its workload and contract. No tests run locally.

In [ ]:
import asyncio, gc, json
from dataclasses import asdict
from importlib.metadata import version
import httpx, torch
from tinyserve.basics import MODEL_ID, load_model
from tinyserve.engine.engine import InferenceEngine
from tinyserve.engine.memory import plan_kv_memory
from tinyserve.generation import generate
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.server.app import create_app
from tinyserve.server.local import serve_local
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_d5_engine.py",
                           "tests/test_e1_server.py", "-v", "-ra"], capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()

## 2. From chat messages to prompt tokens

A request looks like:

```json
{"model":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","messages":[{"role":"user","content":"What is a KV cache?"}],"max_tokens":8,"stream":true}
```

The tokenizer's **chat template** inserts role markers and the assistant prefix. Sending the same words without that template would be a different prompt. We compute direct greedy reference outputs using the identical template.

We load HF weights, copy them strictly into our model, then delete the HF model. This avoids keeping two FP32 reference models throughout the serving demo. Our engine has 16 blocks of 16 slots: 256 processed positions shared by all requests. At 45056 FP32 bytes per position, the fixed pool is **11534336 bytes (11 MiB)**. This is KV storage only, not the model or workspace.

We do not allocate almost all free GPU memory. FP32 uses more model memory than FP16 and keeps this transport lesson separate from cross-precision arithmetic differences; D5 does not claim a universal FP16/FP32 score bound.

For predictable overlapping workloads, the demo temporarily disables EOS stopping: every ordinary request produces eight outputs, and the disconnect case cannot finish early by chance. The server still supports EOS, which the deterministic tests verify. The original EOS setting is restored when the HTTP demo exits, including on failure.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
our_model = LlamaForCausalLM.from_huggingface(hf_model)
del hf_model
gc.collect()
torch.cuda.empty_cache()

original_eos = our_model.config.eos_token_id
our_model.config.eos_token_id = None
messages = [{"role": "user", "content": "Explain a KV cache in one short sentence."}]
other_messages = [{"role": "user", "content": "What does a GPU do?"}]

def chat_ids(chat_messages):
    return tokenizer.apply_chat_template(chat_messages, tokenize=True,
             add_generation_prompt=True, return_tensors="pt", return_dict=False)[0].to(DEVICE)

def reference(chat_messages):
    prompt = chat_ids(chat_messages)
    ids = generate(our_model, prompt, max_new_tokens=8)
    return {"prompt_tokens": prompt.numel(), "token_ids": ids.tolist(),
            "generated_tokens": ids[prompt.numel():].tolist(),
            "text": tokenizer.decode(ids[prompt.numel():], skip_special_tokens=True,
                                     clean_up_tokenization_spaces=False)}

direct_reference = reference(messages)
other_reference = reference(other_messages)
unit = plan_kv_memory(our_model, 1024**2, block_size=16).bytes_per_block
engine = InferenceEngine(our_model, 16 * unit, block_size=16, max_batch_tokens=32)
assert engine.cache.allocated_bytes == 11534336
assert all(result["prompt_tokens"] + 8 <= 256 for result in (direct_reference, other_reference))
app = create_app(engine, tokenizer, MODEL_ID, max_requests=32)
print("KV plan:", asdict(engine.memory_budget))
print("Direct reference:", direct_reference)

## 3. Understand the wire format before reading it

A streaming response stays open. It carries UTF-8 SSE records separated by blank lines. Each record begins `data: `. Most contain a JSON object; successful completion ends with `data: [DONE]`.

| Record | Meaning |
|---|---|
| `delta: {role: assistant, content: ""}` | This is an assistant response; no generated text yet |
| `delta: {content: "some text"}` | Append this text to the response |
| `delta: {}`, `finish_reason: length` or `stop` | Generation finished |
| `[DONE]` | Successful stream termination |

Real JSON quotes its keys; the table is only a diagram. `stop` means the model emitted EOS; `length` means it exhausted the output budget. IDs and timestamps remain stable across one stream.

One event need not equal one token. Transformers' `TextStreamer` buffers incomplete words/Unicode, and slow clients can receive cumulative snapshots that coalesce several tokens. Decode isolated token fragments incorrectly and you can corrupt spaces or Unicode.

The client below reads complete SSE lines, detects error objects and requires the final marker. A stream error is not successful completion, even if its HTTP headers were already 200.

In [ ]:
def request_body(chat_messages, stream=False, max_tokens=8):
    return {"model": MODEL_ID, "messages": chat_messages, "max_tokens": max_tokens,
            "temperature": 0.0, "stream": stream}

async def read_stream(client, body):
    chunks, text_parts, done = [], [], False
    async with client.stream("POST", "/v1/chat/completions", json=body) as response:
        response.raise_for_status()
        assert response.headers["content-type"].startswith("text/event-stream")
        async for line in response.aiter_lines():
            if not line.startswith("data: "):
                continue
            value = line[6:]
            if value == "[DONE]":
                done = True
                break
            chunk = json.loads(value)
            if "error" in chunk:
                raise RuntimeError(chunk["error"]["message"])
            chunks.append(chunk)
            text = chunk["choices"][0]["delta"].get("content", "")
            text_parts.append(text)
            if text:
                print(text, end="", flush=True)
    print()
    assert done and chunks
    assert chunks[0]["choices"][0]["delta"]["role"] == "assistant"
    assert chunks[-1]["choices"][0]["finish_reason"] in ("stop", "length")
    assert all(chunk["id"] == chunks[0]["id"] for chunk in chunks)
    assert all(chunk["created"] == chunks[0]["created"] for chunk in chunks)
    return {"text": "".join(text_parts), "chunks": chunks, "done": done}

## 4. One live server, two concurrent clients and a disconnect

The next cell starts Uvicorn on an OS-assigned `127.0.0.1` port. We await readiness and verify `/health` over actual HTTP. No thread, detached process, fixed port or notebook event-loop patch is needed; Colab supports top-level `await`.

The demo checks:
1. Model discovery and one ordinary JSON completion.
2. SSE text and exact token history against direct greedy generation.
3. Two overlapping HTTP clients sharing the engine. A temporary forward hook records request owners; it does not change model outputs.
4. Unsupported options return HTTP 400 without consuming an engine request.
5. A client closes its stream after the initial role event. We wait for job removal and require all blocks released. The saved finish reason distinguishes cancellation from generation that finished before the disconnect took effect. Reading a role event does not prove inference is still running; the socket tests separately gate inference to verify both outcomes deterministically.

The D5 engine is single-threaded. A forward blocks the event loop; requests join and cancellations take effect only between complete steps. Do not interpret async HTTP as interruptible GPU kernels.

The app's default lifetime quota is 32 admitted requests. This bounds the completed histories D5 retains; exceeding it gives explicit HTTP 429. It is not a production server.

In [ ]:
async def http_demo():
    forward_owners = []
    def record_owners(module, args):
        metadata = args[2]
        if metadata.paged_cache is not None:
            forward_owners.append(metadata.sequence_ids.unique().tolist())
    hook = our_model.register_forward_pre_hook(record_owners)
    try:
        async with serve_local(app) as base_url:
            print("Loopback URL:", base_url)
            async with httpx.AsyncClient(base_url=base_url, timeout=120, trust_env=False) as client:
                health = await client.get("/health")
                assert health.status_code == 200 and health.json()["status"] == "ok"
                listed = await client.get("/v1/models")
                listed.raise_for_status()
                assert listed.json()["data"][0]["id"] == MODEL_ID
                response = await client.post("/v1/chat/completions", json=request_body(messages))
                response.raise_for_status()
                ordinary = response.json()
                assert ordinary["choices"][0]["message"]["content"] == direct_reference["text"]
                assert engine.get_output(0).token_ids == tuple(direct_reference["token_ids"])
                assert ordinary["usage"]["completion_tokens"] == len(direct_reference["generated_tokens"])
                print("Streamed response:")
                streamed = await read_stream(client, request_body(messages, stream=True))
                assert streamed["text"] == direct_reference["text"]
                assert engine.get_output(1).token_ids == tuple(direct_reference["token_ids"])
                concurrent_start = len(forward_owners)
                concurrent = await asyncio.gather(
                    client.post("/v1/chat/completions", json=request_body(messages)),
                    client.post("/v1/chat/completions", json=request_body(other_messages)))
                for reply, expected in zip(concurrent, (direct_reference, other_reference)):
                    reply.raise_for_status()
                    assert reply.json()["choices"][0]["message"]["content"] == expected["text"]
                concurrent_owners = forward_owners[concurrent_start:]
                assert any(len(owners) > 1 for owners in concurrent_owners)
                accepted_before_error = app.state.driver.accepted
                invalid = await client.post("/v1/chat/completions",
                           json={**request_body(messages), "tools": []})
                assert invalid.status_code == 400 and "error" in invalid.json()
                assert app.state.driver.accepted == accepted_before_error
                disconnected_owner = app.state.driver.accepted
                async with client.stream("POST", "/v1/chat/completions",
                         json=request_body(messages, stream=True, max_tokens=128)) as early:
                    early.raise_for_status()
                    async for line in early.aiter_lines():
                        if line.startswith("data: "):
                            first = json.loads(line[6:])
                            assert first["choices"][0]["delta"]["role"] == "assistant"
                            break
                async def wait_for_cleanup():
                    while app.state.driver.jobs:
                        await asyncio.sleep(0.01)
                await asyncio.wait_for(wait_for_cleanup(), timeout=30)
                cancelled = engine.get_output(disconnected_owner)
                assert cancelled.finish_reason in ("cancelled", "length")
                if cancelled.finish_reason == "length":
                    assert len(cancelled.generated_tokens) == 128
                else:
                    assert len(cancelled.generated_tokens) < 128
                assert engine.cache.num_free_blocks == engine.cache.num_blocks
                assert not engine.has_pending_requests
                result = {"health": health.json(), "models": listed.json(),
                          "ordinary": ordinary, "streamed": streamed,
                          "concurrent": [reply.json() for reply in concurrent],
                          "concurrent_forward_owners": concurrent_owners,
                          "invalid_request": {"status": invalid.status_code, "body": invalid.json()},
                          "disconnect": asdict(cancelled),
                          "outputs": [asdict(engine.get_output(owner))
                                      for owner in range(app.state.driver.accepted)],
                          "final_free_blocks": engine.cache.num_free_blocks,
                          "final_tables": dict(engine.cache.block_tables),
                          "forward_owners": forward_owners}
        result["server_stopped"] = True
        try:
            engine.step()
        except RuntimeError as error:
            if "engine is closed" not in str(error):
                raise
            result["closed_engine_error"] = str(error)
        else:
            raise AssertionError("server shutdown did not close its engine")
        return result
    finally:
        hook.remove()
        our_model.config.eos_token_id = original_eos

http_result = await http_demo()
print("Server stopped; free blocks:", http_result["final_free_blocks"])

## 5. Failure and shutdown are part of the API

Before response headers, an inference failure can return HTTP 503. After streaming headers are sent, the HTTP status cannot change. The server sends an SSE `error` object and closes **without `[DONE]`**. The tests inject that failure and check that all waiters are notified, leases are released and future requests get 503.

The driver does not silently retry forever or report a successful finish. `engine.close()` permanently aborts outstanding work, including a failed pending plan. Closing releases block ownership but does not shrink the fixed K/V tensors while the engine is still referenced.

The live demo already stopped its server and closed its engine. To run it again, recreate the engine and app in section 2. Do not restart the same closed engine or retain multiple copies of the model.

## 6. Save complete evidence

The final JSON includes tests, dependency versions, the direct reference, actual HTTP records, concurrent ownership, disconnect cancellation and final release. `save_result()` adds GPU, library versions, source SHA and UTC timestamp.

Token and text-chunk count ranges describe these workloads; they are not performance measurements. E2 will measure throughput, time to first token and latency with explicit timing boundaries.

### Three interview questions
1. Why is SSE not one HTTP response per token? One response carries multiple framed events; token decoding and network buffering can group outputs.
2. When can a disconnected request release its blocks safely? Between completed engine forwards, not halfway through a kernel.
3. How do you signal failure after HTTP 200? Send an explicit stream error and terminate without the success marker.

Guide: `docs/guide/21-streaming-server.md`. Keep the complete JSON and notebook outputs. E1's reviewed `cee6f49` run is archived: JSON/SSE and token parity passed, concurrent clients shared forwards, disconnect cancelled after three tokens, all 16 blocks were released and the server stopped. D5's separate complete run is also archived. Next: E2 benchmarks, then H packaging and a full core-project review before stretch work.

In [ ]:
assert test_run.returncode == 0 and http_result["server_stopped"]
assert http_result["final_free_blocks"] == 16 and http_result["final_tables"] == {}
completed_counts = [reply["usage"]["completion_tokens"] for reply in
                    [http_result["ordinary"], *http_result["concurrent"]]]
text_chunk_sizes = [len(chunk["choices"][0]["delta"].get("content", ""))
                    for chunk in http_result["streamed"]["chunks"]
                    if chunk["choices"][0]["delta"].get("content")]
e1_result = {"test_run": {"returncode": test_run.returncode, "stdout": test_run.stdout,
                          "stderr": test_run.stderr},
             "dtype": "float32", "exposure": "127.0.0.1 only; no public tunnel",
             "dependency_versions": {name: version(name) for name in ("fastapi", "uvicorn", "httpx")},
             "d5_full_model_fp16_validation": "validated at d013743 under paging/greedy-parity contract; no universal cross-precision score bound",
             "memory_budget": asdict(engine.memory_budget), "actual_kv_bytes": engine.cache.allocated_bytes,
             "direct_reference": direct_reference, "other_reference": other_reference,
             "completion_token_count_range": {"min": min(completed_counts), "max": max(completed_counts)},
             "text_chunk_count": len(text_chunk_sizes),
             "text_chunk_character_count_range": ({"min": min(text_chunk_sizes), "max": max(text_chunk_sizes)}
                                                   if text_chunk_sizes else None),
             "http": http_result}
result_path = save_result("E1_streaming_server", e1_result)
print(result_path.read_text())